# Concepts and predictions: correlation or causation?

NAVE's correlation tells which concepts **co-occur** with a class. It doesn't tell whether the model **uses** them:
a background can come with a class without moving its score at all. This notebook asks the causal question with
**interventions**, for a pretrained ResNet50 and a pretrained ViT-S/16:

- NAVE extracts concepts from the encoder; each image's tokens get a concept.
- **Intervention:** at one block, the tokens of a concept are replaced by the tokens at the same positions in
  other images (*resampling ablation*), and the rest of the network runs on. The drop in the true class's score is
  the concept's effect on that image.
- **Control:** the same number of random tokens is replaced the same way. A concept's **causal effect** is its
  effect beyond that control, averaged over the images where it appears, with bootstrap intervals.

Then correlation and causal effect side by side, concept by concept.

**Needs:** `pip install nave-xai torch torchvision timm`. **Downloads:** Imagenette (160 px, about 100 MB) and both
models' ImageNet weights (about 100 MB and 90 MB), once. A GPU is recommended; on a CPU, reduce `N_PER_CLASS`.

In [ ]:
import sys
if "google.colab" in sys.modules:       # on Google Colab: install what this notebook needs
    get_ipython().run_line_magic("pip", 'install -q "nave-xai @ git+https://github.com/Ahcene-B/NAVE.git" timm')
# elsewhere: pip install nave-xai torch torchvision timm

import numpy as np
import torch
import matplotlib.pyplot as plt

MODELS = ("resnet50", "vit")          # either or both
N_PER_CLASS = 10                       # Imagenette validation images per class (10 classes)
K = 10                                 # NAVE concepts per model
DONORS = 4                             # other images whose tokens replace a concept's (resampling ablation)
MIN_TOKENS = 4                         # a concept counts as present in an image from this many tokens (of 196)
BOOT = 1000                            # bootstrap resamples for the intervals
SEED = 0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)

## The images

Imagenette's 10 classes are ImageNet classes, so the pretrained models classify them directly. The images are
resized to 224 x 224 px: both models then have a 14 x 14 grid of tokens at the intervention block.

In [ ]:
from PIL import Image
from torchvision import datasets

IMAGENET_ID = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]     # Imagenette's classes in ImageNet
val = datasets.Imagenette("data", split="val", size="160px", download=True)
class_names = [c[0] if isinstance(c, (tuple, list)) else c for c in val.classes]
targets = np.array([y for _, y in val._samples]) if hasattr(val, "_samples") else np.array([val[i][1] for i in range(len(val))])
chosen = np.concatenate([rng.choice(np.nonzero(targets == c)[0], N_PER_CLASS, replace=False) for c in range(10)])
images = [np.asarray(val[int(i)][0].convert("RGB").resize((224, 224), Image.BICUBIC)) for i in chosen]
labels = targets[chosen]
N = len(images)
print(N, "images:", class_names)

## The models, and where we intervene

- **ResNet50** (torchvision, ImageNet): NAVE on `layer3` and `layer4`; interventions on `layer3`'s output (14 x 14).
- **ViT-S/16** (timm, ImageNet-21k then 1k): NAVE on blocks 8 and 12; interventions on block 8's output
  (14 x 14 patch tokens, the class token left alone).

Interventions on the earlier of the two blocks leave the network room to use, or ignore, what changed.

In [ ]:
import timm
import torchvision
from nave_xai import NAVEImage
from nave_xai.models import TorchBackbone


def load(name):
    if name == "resnet50":
        m = torchvision.models.resnet50(weights="IMAGENET1K_V2").eval().to(DEVICE)
        mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        bb = TorchBackbone(m, mean=mean, std=std, device=DEVICE)
        return dict(name="ResNet50", model=m, backbone=bb, layers=["layer3", "layer4"], block="layer3", prefix=None,
                    mean=mean, std=std)
    m = timm.create_model("vit_small_patch16_224.augreg_in21k_ft_in1k", pretrained=True).eval().to(DEVICE)
    cfg = timm.data.resolve_data_config({}, model=m)                       # this ViT's own normalisation
    mean, std = tuple(cfg["mean"]), tuple(cfg["std"])
    bb = TorchBackbone(m, mean=mean, std=std, multiple=16, fixed_size=(224, 224), prefix_tokens=1,
                       classifier="head", final_norm="norm", device=DEVICE)
    return dict(name="ViT-S/16", model=m, backbone=bb, layers=["blocks.7", "blocks.11"], block="blocks.7", prefix=1,
                mean=mean, std=std)


def to_tensor(imgs, info):
    x = torch.tensor(np.stack(imgs), dtype=torch.float32).permute(0, 3, 1, 2) / 255
    mean = torch.tensor(info["mean"]).view(1, 3, 1, 1); std = torch.tensor(info["std"]).view(1, 3, 1, 1)
    return ((x - mean) / std).to(DEVICE)


models = {name: load(name) for name in MODELS}
print({n: m["name"] for n, m in models.items()})

## Interventions inside the network

A forward hook on the intervention block replaces, image by image, the tokens chosen by a mask with the tokens of a
source (another image's at the same positions), then lets the network run on. Tokens are handled as
(tokens, channels) whatever the architecture: a ResNet's (C, 14, 14) map, or a ViT's patch tokens after its class
token.

In [ ]:
def tokens_of(out, info):
    """The block's output as (batch, 196, channels)."""
    return out.flatten(2).transpose(1, 2) if info["prefix"] is None else out[:, info["prefix"]:]


def with_tokens(out, tokens, info):
    """The block's output with its 196 tokens replaced by `tokens` (batch, 196, channels)."""
    if info["prefix"] is None:
        return tokens.transpose(1, 2).reshape(out.shape)
    return torch.cat([out[:, :info["prefix"]], tokens], dim=1)


@torch.no_grad()
def run(info, imgs, masks=None, sources=None, keep=None):
    """Logits of `imgs`; with masks (batch, 196) and sources (batch, 196, channels), the masked tokens at the
    intervention block are replaced by the sources'. keep=list collects the clean block tokens."""
    block = dict(info["model"].named_modules())[info["block"]]
    def hook(module, inputs, out):
        t = tokens_of(out, info)
        if keep is not None:
            keep.append(t.detach().cpu())
        if masks is None:
            return None
        m = torch.as_tensor(masks, device=out.device)[..., None]
        return with_tokens(out, torch.where(m, torch.as_tensor(sources, device=out.device, dtype=t.dtype), t), info)
    handle = block.register_forward_hook(hook)
    try:
        return info["model"](to_tensor(imgs, info)).cpu().numpy()
    finally:
        handle.remove()


for info in models.values():                       # clean logits and block tokens of every image
    keep, logits = [], []
    for b in range(0, N, 25):
        logits.append(run(info, images[b:b + 25], keep=keep))
    info["logits"], info["acts"] = np.concatenate(logits), torch.cat(keep).numpy()
    pred = info["logits"].argmax(1)
    info["accuracy"] = float(np.mean(pred == np.array(IMAGENET_ID)[labels]))
    print(f"{info['name']}: accuracy on these images {info['accuracy']:.2f}, block tokens {info['acts'].shape}")

## NAVE's concepts, token by token

NAVE fits K concepts on the images; a token's concept is the one at its cell's centre in the explanation map.

In [ ]:
centres = np.arange(8, 224, 16)
for info in models.values():
    nave = NAVEImage(info["backbone"], layers=info["layers"], k=K, seed=SEED).fit(images)
    maps = [e.explanation for e in nave.predict(images)]
    info["nave"], info["maps"] = nave, maps
    info["token_concept"] = np.stack([m[np.ix_(centres, centres)].ravel() for m in maps])       # (N, 196)
    counts = np.stack([np.bincount(t, minlength=K) for t in info["token_concept"]])
    info["proportions"] = counts / 196
    print(info["name"], "images per concept:", (counts >= MIN_TOKENS).sum(0))

## Correlation: which concepts come with which class

The Pearson correlation, over the images, between a concept's share of an image and the image's class.

In [ ]:
def correlation(proportions, labels, n_classes=10):
    Y = (labels[:, None] == np.arange(n_classes)[None]).astype(float)
    P = proportions - proportions.mean(0); Y = Y - Y.mean(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        R = (P.T @ Y) / (np.sqrt((P ** 2).sum(0))[:, None] * np.sqrt((Y ** 2).sum(0))[None])
    return np.nan_to_num(R)                                                  # (K, classes)

for info in models.values():
    info["corr"] = correlation(info["proportions"], labels)
    fig, ax = plt.subplots(figsize=(7, 3.2))
    im = ax.imshow(info["corr"], cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(10), [n[:10] for n in class_names], rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("concept"); ax.set_title(f"{info['name']}: correlation of concepts and classes", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.03); plt.show()

## Intervention: what happens to the score without a concept

For every image and every concept present in it (at least `MIN_TOKENS` tokens):
- **concept:** the concept's tokens get the tokens of `DONORS` other images at the same positions;
- **control:** as many random tokens, outside the concept, get the same treatment.

The effect is the drop in the logit of the image's true class, averaged over the donors.

In [ ]:
records = []
for key, info in models.items():
    acts, tc, clean = info["acts"], info["token_concept"], info["logits"]
    for i in range(N):
        y = IMAGENET_ID[labels[i]]
        donors = rng.choice([j for j in range(N) if j != i], DONORS, replace=False)
        for c in range(K):
            mask = tc[i] == c
            n = int(mask.sum())
            if n < MIN_TOKENS:
                continue
            outside = np.nonzero(~mask)[0]
            random_masks = []
            for _ in donors:
                r = np.zeros(196, bool); r[rng.choice(outside, min(n, len(outside)), replace=False)] = True
                random_masks.append(r)
            masks = np.stack([mask] * DONORS + random_masks)
            sources = np.concatenate([acts[donors], acts[donors]])
            out = run(info, [images[i]] * (2 * DONORS), masks, sources)
            drop = clean[i, y] - out[:, y]
            records.append(dict(model=key, image=i, label=labels[i], concept=c, tokens=n,
                                concept_drop=float(drop[:DONORS].mean()), control_drop=float(drop[DONORS:].mean())))
    print(info["name"], "done:", sum(r["model"] == key for r in records), "interventions (each with its control)")

## The causal effect of each concept on each class

For a concept and a class: the mean, over that class's images where the concept is present, of the concept's drop
minus its control's drop, with a 95% bootstrap interval over the images. Pairs seen in fewer than 3 images are left
out.

In [ ]:
def bootstrap(values, boot=BOOT):
    values = np.asarray(values)
    means = values[rng.integers(0, len(values), (boot, len(values)))].mean(1)
    return values.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

effects = {}
for key, info in models.items():
    rows = []
    for c in range(K):
        for y in range(10):
            v = [r["concept_drop"] - r["control_drop"] for r in records if r["model"] == key and r["concept"] == c and r["label"] == y]
            if len(v) >= 3:
                m, lo, hi = bootstrap(v)
                rows.append(dict(concept=c, label=y, n=len(v), effect=m, lo=lo, hi=hi, corr=info["corr"][c, y]))
    effects[key] = rows
    print(f"{info['name']}: {len(rows)} concept-class pairs")

## Correlation against causal effect

Each point is a concept and a class. Right: the concept comes with the class; up: removing it lowers the class's
score beyond the control. The two axes need not agree: a concept can come with a class without being used (right,
low), or be used without being specific to the class (left, high).

In [ ]:
from scipy.stats import spearmanr

fig, axes = plt.subplots(1, len(models), figsize=(6.2 * len(models), 4.6), squeeze=False)
for ax, (key, info) in zip(axes[0], models.items()):
    rows = effects[key]
    x = np.array([r["corr"] for r in rows]); y = np.array([r["effect"] for r in rows])
    err = np.array([[r["effect"] - r["lo"] for r in rows], [r["hi"] - r["effect"] for r in rows]])
    colours = info["nave"].palette[[r["concept"] for r in rows]] / 255
    ax.errorbar(x, y, yerr=err, fmt="none", ecolor="0.75", lw=1, zorder=1)
    ax.scatter(x, y, c=colours, s=36, edgecolors="black", linewidths=0.4, zorder=2)
    ax.axhline(0, color="0.5", lw=0.8); ax.axvline(0, color="0.5", lw=0.8)
    rho = spearmanr(x, y).statistic
    ax.set_xlabel("correlation of the concept with the class"); ax.set_ylabel("causal effect on the class's logit")
    ax.set_title(f"{info['name']}  (Spearman correlation {rho:.2f})", fontsize=10)
fig.tight_layout()

### Where they disagree

The pairs that are most correlated with little effect, and those with the most effect for little correlation, with
an example image of each: the concept in colour, the rest dimmed.

In [ ]:
def show_pair(info, row, ax):
    imgs = [r["image"] for r in records if r["model"] == key and r["concept"] == row["concept"] and r["label"] == row["label"]]
    i = imgs[0]
    m = info["maps"][i] == row["concept"]
    img = images[i].astype(float) / 255
    ax.imshow(np.where(m[..., None], img, 0.25 + 0.2 * img)); ax.axis("off")
    ax.set_title(f"concept {row['concept']}, {class_names[row['label']][:14]}\n"
                 f"correlation {row['corr']:.2f}, effect {row['effect']:.2f}", fontsize=8)

for key, info in models.items():
    rows = effects[key]
    used_not_specific = sorted(rows, key=lambda r: r["corr"] - r["effect"] / (abs(max(q["effect"] for q in rows)) + 1e-9))[:3]
    specific_not_used = sorted(rows, key=lambda r: r["effect"] / (abs(max(q["effect"] for q in rows)) + 1e-9) - r["corr"])[:3]
    fig, axes = plt.subplots(1, 6, figsize=(15, 3.2))
    for ax, row in zip(axes, specific_not_used + used_not_specific):
        show_pair(info, row, ax)
    fig.suptitle(f"{info['name']}: correlated but little effect (left three), effect without correlation (right three)", fontsize=10)
    plt.show()

## A contrast: intervening on the pixels

The same question in pixel space: the concept's region is greyed out (the image's mean colour), and the drop in the
true class's logit is compared with the feature-space effect, image by image. Pixel edits produce images the model
has never seen, so part of their effect comes from the edit itself.

In [ ]:
for key, info in models.items():
    feats, pixels = [], []
    subset = [r for r in records if r["model"] == key][:400]
    for b in range(0, len(subset), 50):
        batch = subset[b:b + 50]
        edited = []
        for r in batch:
            img = images[r["image"]].copy()
            m = info["maps"][r["image"]] == r["concept"]
            img[m] = img.reshape(-1, 3).mean(0).astype(np.uint8)
            edited.append(img)
        out = run(info, edited)
        for r, o in zip(batch, out):
            y = IMAGENET_ID[r["label"]]
            pixels.append(info["logits"][r["image"], y] - o[y]); feats.append(r["concept_drop"])
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    ax.scatter(feats, pixels, s=10, alpha=0.6)
    lim = [min(min(feats), min(pixels)), max(max(feats), max(pixels))]
    ax.plot(lim, lim, color="0.5", lw=0.8)
    ax.set_xlabel("drop, feature intervention"); ax.set_ylabel("drop, pixel intervention")
    ax.set_title(f"{info['name']}  (Spearman {spearmanr(feats, pixels).statistic:.2f})", fontsize=10); plt.show()

## One image, concept by concept

An image's concepts coloured by their effect on its class: the concept's drop minus its control's, for this image.

In [ ]:
IMAGE = 0                                                                  # any index below N
fig, axes = plt.subplots(len(models), 3, figsize=(11, 3.7 * len(models)), squeeze=False)
for row_axes, (key, info) in zip(axes, models.items()):
    m = info["maps"][IMAGE]
    eff = np.full(K, np.nan)
    for r in records:
        if r["model"] == key and r["image"] == IMAGE:
            eff[r["concept"]] = r["concept_drop"] - r["control_drop"]
    lim = np.nanmax(np.abs(eff)) or 1
    row_axes[0].imshow(images[IMAGE]); row_axes[0].set_title(class_names[labels[IMAGE]], fontsize=9)
    row_axes[1].imshow(info["nave"].palette[m]); row_axes[1].set_title(f"{info['name']}: NAVE concepts", fontsize=9)
    shown = row_axes[2].imshow(eff[m], cmap="RdBu_r", vmin=-lim, vmax=lim)
    row_axes[2].set_title("causal effect on its class (beyond the control)", fontsize=9)
    plt.colorbar(shown, ax=row_axes[2], fraction=0.046)
    for ax in row_axes:
        ax.axis("off")
fig.tight_layout()

## What the effect means, and what it doesn't

- **Causal for the model, not for the world.** The effect measures how much the model's score depends on a concept's
  features. It says nothing about whether the concept causes the class in reality.
- **Interventions shift the distribution.** Replacing tokens with other images' keeps them realistic one by one, but
  their combination may still be unusual for the network. The control (as many random tokens) absorbs part of this,
  not all.
- **Effects don't add up.** Concepts interact: removing two together can do more, or less, than the sum.
- **The block matters.** An intervention early leaves later blocks the chance to recover the information from other
  tokens; a ViT's attention does that readily.

## Things to look at

- concepts strongly correlated with a class but with no effect: context the model ignores;
- concepts with an effect on several classes: shared features rather than class evidence;
- the ResNet against the ViT: which relies more on a few concepts?
- another intervention block (`block` in `load`), or other replacements (the mean token instead of donors).

## Citation

If you use nave-xai, please cite the paper:

```bibtex
@inproceedings{boubekki2026nave,
  author    = {Boubekki, Ahc{\`e}ne and Fadel, Samuel G. and Mair, Sebastian},
  title     = {Multi-depth Concept Extraction for Post-Hoc Vision Encoder Explanation},
  booktitle = {Machine Learning and Knowledge Discovery in Databases. Research Track},
  publisher = {Springer Nature Switzerland},
  address   = {Cham},
  pages     = {449--467},
  year      = {2026},
  doi       = {10.1007/978-3-032-37664-0_26}
}
```